In [ ]:
!pip install ultralytics

In [12]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

# Update this path to where your zip is in Google Drive
zip_path = '/content/drive/MyDrive/thesis/videoAnalysis/videoAnalysis.zip'

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('dataset_raw')

print("Files extracted:")
!ls dataset_raw/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files extracted:
__MACOSX  videoAnalysis


In [13]:
import os, shutil, random
from pathlib import Path

raw = Path('dataset_raw')

all_images = list(raw.rglob('*.jpg')) + list(raw.rglob('*.png'))
all_labels = list(raw.rglob('*.txt'))

# Label stems have %3A instead of colon — normalize both to underscore
label_lookup = {l.stem.replace('%3A', '_').replace(':', '_'): l for l in all_labels}

# Clean up old dataset
if Path('dataset').exists():
    shutil.rmtree('dataset')

random.seed(42)
random.shuffle(all_images)
split      = int(len(all_images) * 0.8)
train_imgs = all_images[:split]
val_imgs   = all_images[split:]

matched = 0
for split_name, imgs in [('train', train_imgs), ('val', val_imgs)]:
    Path(f'dataset/images/{split_name}').mkdir(parents=True, exist_ok=True)
    Path(f'dataset/labels/{split_name}').mkdir(parents=True, exist_ok=True)
    for img in imgs:
        clean_stem = img.stem.replace(':', '_')
        clean_name = clean_stem + img.suffix
        shutil.copy(img, f'dataset/images/{split_name}/{clean_name}')
        lbl = label_lookup.get(clean_stem)
        if lbl:
            shutil.copy(lbl, f'dataset/labels/{split_name}/{clean_stem}.txt')
            matched += 1

print(f"Train: {len(train_imgs)} images")
print(f"Val:   {len(val_imgs)} images")
print(f"Labels matched: {matched} / {len(all_images)}")

Train: 576 images
Val:   144 images
Labels matched: 720 / 720


In [14]:
yaml_content = """
path: /content/dataset
train: images/train
val: images/val

nc: 1
names: ['character']
"""
with open('dataset.yaml', 'w') as f:
    f.write(yaml_content)
print("dataset.yaml created")

dataset.yaml created


In [15]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='character_detector',
    patience=10,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7